In [ ]:
#@title 8. 结果可视化

# 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 损失曲线
axes[0].plot(train_losses, marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('训练损失')
axes[0].grid(True, alpha=0.3)

# 准确率曲线
axes[1].plot(train_accuracies, marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('训练准确率')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 实验总结
print(f"\\n{'='*60}")
print("实验总结")
print(f"{'='*60}")
print(f"数据集: {DATASET}")
print(f"Coreset方法: {CORESET_METHOD}")
print(f"摘要比例: {COSET_RATIO*100:.1f}%")
print(f"摘要大小: {len(coreset_data)} / {len(full_data)}")
print(f"Coreset选择时间: {selection_time:.2f}秒")
print(f"最终测试准确率: {test_accuracy:.2f}%")
print(f"{'='*60}")

In [ ]:
#@title 7.2 在测试集上评估

model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for X, y in tqdm(test_loader, desc='Testing'):
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        _, predicted = outputs.max(1)
        test_total += y.size(0)
        test_correct += predicted.eq(y).sum().item()

test_accuracy = 100. * test_correct / test_total
print(f"\\n{'='*60}")
print(f"测试集准确率: {test_accuracy:.2f}%")
print(f"{'='*60}")

In [ ]:
#@title 7.1 在Coreset上训练模型

# 创建模型
model = SimpleCNN(in_channels=1 if DATASET in ["MNIST", "Fashion-MNIST"] else 3, 
                  num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 准备训练数据
from torch.utils.data import TensorDataset
train_dataset_small = TensorDataset(
    torch.FloatTensor(coreset_data).view(-1, 1 if DATASET in ["MNIST", "Fashion-MNIST"] else 3, 28, 28),
    torch.LongTensor(coreset_labels)
)
train_loader_small = DataLoader(train_dataset_small, batch_size=BATCH_SIZE, shuffle=True)

# 训练
print(f"在Coreset上训练模型 ({len(coreset_data)}个样本)...")
train_losses = []
train_accuracies = []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader_small, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for X, y in pbar:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    epoch_loss = running_loss / len(train_loader_small)
    epoch_acc = 100. * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    print(f'Epoch {epoch+1}: Loss={epoch_loss:.4f}, Acc={epoch_acc:.2f}%')

print("✓ 训练完成")

## 7. 模型训练与评估

在部分数据集上训练模型并评估性能。

In [ ]:
#@title 6. 可视化Coreset选择结果

# 可视化前20个选中的样本
fig, axes = plt.subplots(2, 10, figsize=(20, 4))
axes = axes.flatten()

for i in range(min(20, len(selected_indices))):
    idx = selected_indices[i]
    if DATASET in ["MNIST", "Fashion-MNIST"]:
        img = full_data[idx].reshape(28, 28)
        axes[i].imshow(img, cmap='gray')
    else:  # CIFAR-10
        img = full_data[idx].reshape(3, 32, 32).transpose(1, 2, 0)
        axes[i].imshow(img)
    
    axes[i].set_title(f"Idx: {idx}\\nLabel: {full_labels[idx]}")
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle(f"前20个选中的样本 ({CORESET_METHOD})", y=1.05, fontsize=14)
plt.show()

# 显示类别分布
unique, counts = np.unique(coreset_labels, return_counts=True)
plt.figure(figsize=(10, 5))
plt.bar(unique, counts)
plt.xlabel('类别')
plt.ylabel('样本数')
plt.title(f'Coreset类别分布 ({CORESET_METHOD})')
plt.xticks(unique)
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
#@title 5. 应用Coreset方法

# 计算摘要大小
coreset_size = int(len(full_data) * COSET_RATIO)
print(f"正在应用 {CORESET_METHOD} 方法...")
print(f"目标摘要大小: {coreset_size} ({COSET_RATIO*100:.1f}%)")

# 获取coreset方法
method = get_coreset_method(CORESET_METHOD, coreset_size)

# 应用coreset选择并计时
start_time = time.time()
selected_indices, coreset_data, coreset_labels = method.select(full_data, full_labels)
selection_time = time.time() - start_time

print(f"✓ Coreset选择完成")
print(f"  实际摘要大小: {len(selected_indices)}")
print(f"  选择时间: {selection_time:.2f}秒")

In [ ]:
#@title 4. 加载数据集

device = get_device()

# 加载数据集
print(f"正在加载 {DATASET} 数据集...")
train_dataset, test_dataset = get_dataset(DATASET)

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 获取完整训练数据
for X, y in train_loader:
    full_data = X.view(X.size(0), -1).numpy()
    full_labels = y.numpy()
    break

print(f"✓ 数据集加载完成")
print(f"  训练集大小: {len(train_dataset)}")
print(f"  测试集大小: {len(test_dataset)}")
print(f"  数据维度: {full_data.shape[1]}")

In [ ]:
#@title 3. 实验参数设置

#@markdown 选择数据集
DATASET = "MNIST"  #@param ["MNIST", "Fashion-MNIST", "CIFAR-10"]

#@markdown Coreset方法
CORESET_METHOD = "K-Center"  #@param ["Random", "K-Center", "Herding", "Greedy k-Center", "Sensitivity Sampling"]

#@markdown 摘要大小（原始数据的百分比）
COSET_RATIO = 0.1  #@param {type:"slider", min:0.01, max:0.5, step:0.01}

#@markdown 训练轮数
NUM_EPOCHS = 10  #@param {type:"integer"}

#@markdown 批次大小
BATCH_SIZE = 128  #@param {type:"integer"}

# 显示配置
print("=" * 60)
print("实验配置")
print("=" * 60)
print(f"数据集: {DATASET}")
print(f"Coreset方法: {CORESET_METHOD}")
print(f"摘要比例: {COSET_RATIO*100:.1f}%")
print(f"训练轮数: {NUM_EPOCHS}")
print(f"批次大小: {BATCH_SIZE}")
print("=" * 60)

In [ ]:
#@title 2. 导入项目模块

# 添加项目路径
import sys
from pathlib import Path

def get_project_path():
    """自动检测项目路径"""
    if 'google.colab' in sys.modules:
        # Colab环境
        project_path = Path('/content/coreset_benchmark')
    else:
        # 本地环境
        project_path = Path().absolute().parent
    
    # 添加到sys.path
    if str(project_path) not in sys.path:
        sys.path.insert(0, str(project_path))
    
    return project_path

project_root = get_project_path()
print(f"项目根目录: {project_root}")

# 导入项目模块
from src.data import get_dataset
from src.coreset import get_coreset_method
from src.models import SimpleCNN
from src.utils import set_seed, get_device

print("✓ 项目模块导入完成")

In [ ]:
#@title 1. 环境设置
#@markdown 运行此单元格以安装必要的依赖

import os
import sys

# 在Colab中运行设置脚本
if 'google.colab' in sys.modules:
    print("在Colab环境中运行...")
    # !wget https://raw.githubusercontent.com/yourusername/coreset_benchmark/main/notebooks/setup_colab.py -O setup_colab.py
    # !python setup_colab.py
else:
    print("在本地环境中运行...")
    print("请确保已运行 setup_colab.py 或手动安装了所有依赖")

# 导入必要的库
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import time
from pathlib import Path

# 设置随机种子
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n✓ 环境设置完成")
print(f"  PyTorch版本: {torch.__version__}")
print(f"  CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU设备: {torch.cuda.get_device_name(0)}")

# 数据摘要实验 (Data Summarization Experiment)

本notebook用于测试不同coreset方法在数据摘要任务中的性能。

## 实验内容

1. **数据集选择**: MNIST / Fashion-MNIST / CIFAR-10
2. **Coreset方法**: 
   - Random Sampling (随机采样)
   - K-Center (K中心)
   - Herding (聚类)
   - Greedy k-Center (贪婪K中心)
   - Sensitivity Sampling (敏感度采样)
3. **评估指标**:
   - 重建误差
   - 分类准确率
   - 运行时间

## 使用说明

1. 运行第一个单元格设置环境
2. 使用交互式控件调整参数
3. 运行实验并查看结果